# Missing-address candidate rescue with name-only embeddings
This experiment adds a small, independent retrieval channel over **all training S2/S3 records whose address is blank**, not just known matches. It searches this pool for every sampled S1 business. Ground truth is used only after retrieval to measure accuracy.

Baseline: combined-text embeddings, FAISS nprobe=256, overfetch=200, final 20. Add name-only top 5/10/20, deduplicate, and measure recall and the saved classifier at its fixed standard-profile cutoff. Also test a fixed-20 allocation (15 baseline + 5 name candidates, filling duplicate slots from baseline). The name-only cosine is **not substituted** for the original combined-text cosine feature: all pairs retain the model's existing 19-feature definition.

Use the same attached original full_pool_embedding_v1 and upgraded training output (your ml-needed dataset). GPU T4 and Internet are needed to load the same pinned public E5 weights and encode the small missing-address pool. No external business lookup occurs. No model is fine-tuned, no full-pool embeddings are regenerated, and no submission is produced.

The fixed 500-business tuning sample allows direct comparison with the gap audit. Results are exploratory tuning results, not independent evaluation. Save the completed notebook version and its outputs.


In [ ]:
%pip install -q numpy==2.0.2 pandas==2.3.3 scikit-learn==1.6.1 faiss-cpu==1.11.0 threadpoolctl transformers==5.0.0 safetensors sentencepiece


In [ ]:
from pathlib import Path
import os, sys, json, pickle, shutil, time, hashlib, gc, zipfile
import numpy as np
import pandas as pd
from threadpoolctl import threadpool_limits

# Set explicit paths if more than one matching saved output is attached.
UPGRADE_PATH = ""
CACHE_PATH = ""
N_TUNE = 500
BATCH_SIZE = 64
ROOT = Path('/kaggle/working/name_only_rescue_v1')
OUT = ROOT / 'results'
OUT.mkdir(parents=True, exist_ok=True)

def discover(explicit, marker, required):
    paths = [Path(explicit)] if explicit else sorted({p.parent for p in Path('/kaggle/input').rglob(marker)})
    paths = [p for p in paths if all((p / n).exists() for n in required)]
    assert len(paths) == 1, f'Set explicit path; found {paths}'
    return paths[0]

UPGRADE = discover(UPGRADE_PATH, 'upgrade_report.json',
    ['sample.pkl', 'queries.npy', 'model/matcher.pkl', 'model/experiment_results.json'])
SOURCE = discover(CACHE_PATH, 'encoding_complete.json',
    ['vectors.f16', 'records.sqlite', 'index_catalog.json', 'sample.tsv', 'configuration.json'])
SAVED_CODE = UPGRADE.parent / 'code'
assert (SAVED_CODE/'submit_pipeline.py').exists()
sys.path.insert(0, str(SAVED_CODE))
import embedding_core as c
import submit_pipeline as s
bundle, model_meta = s.load_model(UPGRADE/'model')
report = json.loads((UPGRADE/'upgrade_report.json').read_text())
threshold = float(report['tuning_options']['standard']['threshold'])
with (UPGRADE/'sample.pkl').open('rb') as f:
    sample, truth = pickle.load(f)
queries = np.load(UPGRADE/'queries.npy', mmap_mode='r')
eligible = np.flatnonzero(sample['split'].to_numpy() == 'tune')
chosen = np.random.default_rng(20260928).choice(eligible, min(N_TUNE,len(eligible)), replace=False)
frame = sample.iloc[chosen].reset_index(drop=True)
q = np.ascontiguousarray(queries[chosen], dtype=np.float32)
assert np.allclose(np.linalg.norm(q,axis=1),1,atol=1e-4)
assert len(frame) and frame.entity_id.is_unique
frame.to_csv(OUT/'audit_businesses.tsv',sep='\t',index=False)
print('Saved upgrade:',UPGRADE)
print('Training cache:',SOURCE)
print('Tuning businesses:',len(frame),'fixed cutoff:',threshold)


In [ ]:
# Copy immutable cache files locally; never alter the attached source.
prior_caches = [Path('/kaggle/working/entity_gap_audit_v1/local_cache'), Path('/kaggle/working/faiss_search_ablation_v1/local_cache')]
CACHE = next((p for p in prior_caches if (p/'copy_identity.json').exists()), ROOT/'local_cache')
CACHE.mkdir(exist_ok=True)
catalog = json.loads((SOURCE/'index_catalog.json').read_text())
names = ['vectors.f16','records.sqlite','encoding_complete.json','index_catalog.json'] + [x['file'] for x in catalog]
manifest = {'source':str(SOURCE), 'sizes':{n:(SOURCE/n).stat().st_size for n in names}}
identity = CACHE/'copy_identity.json'
if identity.exists():
    assert json.loads(identity.read_text()) == manifest, 'Different source cache; use a new ROOT.'
else:
    assert not any(CACHE.iterdir()), 'Unknown files in cache directory; use a new ROOT.'
    identity.write_text(json.dumps(manifest))
need = sum(manifest['sizes'][n] for n in names if not (CACHE/n).exists())
assert shutil.disk_usage(CACHE).free > need + 2*2**30, 'Need enough disk for cache plus 2 GiB.'
for name in names:
    dest=CACHE/name
    if dest.exists():
        assert dest.stat().st_size == manifest['sizes'][name]
        continue
    print('Copying',name,flush=True)
    temp=dest.with_name(dest.name+'.partial')
    shutil.copyfile(SOURCE/name,temp)
    assert temp.stat().st_size == manifest['sizes'][name]
    os.replace(temp,dest)
print('Local cache ready.')


In [ ]:
import torch, faiss
assert torch.cuda.is_available(), 'Enable a GPU before running this name-embedding experiment.'
searcher=s.TestSearcher(CACHE,256,200)
faiss.omp_set_num_threads(4)
POOL=ROOT/'name_pool'
POOL.mkdir(exist_ok=True)
config={'model':c.MODEL_ID,'revision':c.REVISION,'text':'query: NAME','max_length':256,
        'cache_source':str(SOURCE),'cache_rows':searcher.total,
        'sample_indices':chosen.tolist(),'threshold':threshold,'batch_size':BATCH_SIZE,
        'core_sha256':hashlib.sha256((SAVED_CODE/'embedding_core.py').read_bytes()).hexdigest(),
        'matcher_sha256':hashlib.sha256((UPGRADE/'model/matcher.pkl').read_bytes()).hexdigest()}
cp=ROOT/'configuration.json'
if cp.exists():assert json.loads(cp.read_text())==config, 'Config changed; use a new ROOT.'
else:c.atomic_json(cp,config)

# Scan supplied records only; no ground-truth filtering.
pool_file=POOL/'records.pkl'
if pool_file.exists():
    missing_pool=pd.read_pickle(pool_file)
else:
    collected=[];count=0
    cursor=searcher.db.execute('SELECT rid,entity_id,business_name,business_address,country FROM records ORDER BY rid')
    while True:
        batch=cursor.fetchmany(100000)
        if not batch:break
        for row in batch:
            if row[3] is None or not str(row[3]).strip(): collected.append(row)
        count+=len(batch)
        if count%1000000==0:print(f'Scanned {count:,}; blank-address pool {len(collected):,}',flush=True)
    missing_pool=pd.DataFrame(collected,columns=['rid']+c.COLS)
    missing_pool['business_address']=missing_pool.business_address.fillna('')
    temp=pool_file.with_suffix('.partial');missing_pool.to_pickle(temp);os.replace(temp,pool_file)
assert len(missing_pool)>0 and missing_pool.rid.is_unique and missing_pool.entity_id.is_unique
assert missing_pool.business_address.map(lambda x:not str(x).strip()).all()
print('Name-search pool:',len(missing_pool),'S2/S3 records')
print(missing_pool.country.value_counts().to_dict())
assert shutil.disk_usage(ROOT).free > len(missing_pool)*384*4 + 1024**3, 'Insufficient disk.'


In [ ]:
# Same encoder/weights, but name-only text on BOTH sides.
encoder=c.Encoder(BATCH_SIZE)
def encode_names(dataframe):
    texts=c.text_view(dataframe,'name')
    parts=[]
    with torch.inference_mode():
        for start in range(0,len(texts),BATCH_SIZE):
            inputs=encoder.tokenizer(texts[start:start+BATCH_SIZE],max_length=256,
                padding=True,truncation=True,return_tensors='pt').to('cuda')
            hidden=encoder.model(**inputs).last_hidden_state.float()
            mask=inputs['attention_mask'].unsqueeze(-1)
            vectors=(hidden*mask).sum(1)/mask.sum(1).clamp(min=1)
            vectors=torch.nn.functional.normalize(vectors,p=2,dim=1)
            parts.append(vectors.cpu().numpy())
    return np.concatenate(parts).astype(np.float32)

parts_dir=POOL/'embedding_parts';parts_dir.mkdir(exist_ok=True)
chunk_size=5000
for start in range(0,len(missing_pool),chunk_size):
    end=min(start+chunk_size,len(missing_pool))
    dest=parts_dir/f'{start:09d}.npy'
    if dest.exists():
        assert np.load(dest,mmap_mode='r').shape==(end-start,c.DIM)
        continue
    vectors=encode_names(missing_pool.iloc[start:end])
    with dest.with_suffix('.partial').open('wb') as f:np.save(f,vectors)
    os.replace(dest.with_suffix('.partial'),dest)
    print(f'Name embeddings {end:,}/{len(missing_pool):,}',flush=True)
query_path=POOL/'query_names.npy'
if query_path.exists(): name_q=np.load(query_path)
else:
    name_q=encode_names(frame)
    with query_path.with_suffix('.partial').open('wb') as f:np.save(f,name_q)
    os.replace(query_path.with_suffix('.partial'),query_path)
assert name_q.shape==(len(frame),c.DIM)
del encoder;gc.collect();torch.cuda.empty_cache()
name_vectors=np.concatenate([np.load(parts_dir/f'{start:09d}.npy') for start in range(0,len(missing_pool),chunk_size)])
assert name_vectors.shape==(len(missing_pool),c.DIM)
print('Name vectors ready.')


In [ ]:
# This pool is small enough for exact name-vector search, partitioned by country.
# Production would need separate scaling measurement; this is not full S2/S3 exact search.
name_ids=np.full((len(frame),20),-1,np.int64)
name_scores=np.full((len(frame),20),-np.inf,np.float32)
retrieval_timings=[]
with threadpool_limits(limits=4):
    for country in sorted(set(frame.country.map(c.norm))):
        positions=np.flatnonzero(missing_pool.country.map(c.norm).to_numpy()==country)
        owners=np.flatnonzero(frame.country.map(c.norm).to_numpy()==country)
        if not len(positions):continue
        t=time.perf_counter()
        index=faiss.IndexFlatIP(c.DIM)
        target=np.ascontiguousarray(name_vectors[positions],dtype=np.float32)
        faiss.normalize_L2(target);index.add(target)
        index_seconds=time.perf_counter()-t
        t=time.perf_counter()
        scores,local=index.search(np.ascontiguousarray(name_q[owners],dtype=np.float32),20)
        for j,i in enumerate(owners):
            valid=local[j]>=0
            name_ids[i,valid]=missing_pool.rid.to_numpy()[positions[local[j,valid]]]
            name_scores[i,valid]=scores[j,valid]
        retrieval_timings.append(dict(country=country,pool_rows=len(positions),query_rows=len(owners),
             index_seconds=index_seconds,search_seconds=time.perf_counter()-t))
        del index,target
    t=time.perf_counter()
    baseline_ids,baseline_cos=searcher.search(frame,q)
    baseline_seconds=time.perf_counter()-t
pd.DataFrame(retrieval_timings).to_csv(OUT/'name_search_timings.csv',index=False)
print('Baseline search seconds:',round(baseline_seconds,2))
display(pd.DataFrame(retrieval_timings))


In [ ]:
def unique_ids(values):
    return list(dict.fromkeys(int(x) for x in values if x>=0))

policies={'baseline20':[unique_ids(row) for row in baseline_ids]}
for extra in (5,10,20):
    policies[f'baseline20_plus_name{extra}']=[unique_ids(list(baseline_ids[i])+list(name_ids[i,:extra])) for i in range(len(frame))]
policies['budget20_15base_5name']=[unique_ids(list(baseline_ids[i,:15])+list(name_ids[i,:5])+list(baseline_ids[i]))[:20] for i in range(len(frame))]

all_ids=np.asarray(sorted(set(x for values in policies.values() for row in values for x in row)),np.int64)
records=searcher.fetch(all_ids)
features=[];slots=[];combined={};name_lookup={}
for i,left in enumerate(frame[c.COLS].itertuples(index=False,name=None)):
    union=policies['baseline20_plus_name20'][i]
    for rid in union:
        right=records[rid]
        assert c.norm(left[3])==c.norm(right[3])
        v=np.asarray(searcher.vectors[rid],np.float32)
        cosine=float(v.dot(q[i])/max(np.linalg.norm(v),1e-12))
        combined[i,rid]=cosine
        # Important: model still receives ORIGINAL combined-text cosine, not name-only cosine.
        features.append(c.features(left,right)+[cosine]);slots.append((i,rid))
    for j,rid in enumerate(name_ids[i]):
        if rid>=0:name_lookup[i,int(rid)]=float(name_scores[i,j])
with threadpool_limits(limits=4):
    predictions=bundle['model'].predict_proba(np.asarray(features,np.float32))[:,1]
probability=dict(zip(slots,map(float,predictions)))

summaries=[];slices=[];link_audit=[];fp_rows=[]
for policy,lists in policies.items():
    scores=[];tp=fp=found=total=singletons=singleton_correct=0
    blank_total=blank_found=blank_accepted=0
    for i,sid in enumerate(frame.entity_id):
        expected=truth[sid]
        candidates={records[rid][0] for rid in lists[i]}
        accepted={records[rid][0] for rid in lists[i] if probability[i,rid]>=threshold}
        t=len(expected&accepted);f=len(accepted-expected);miss=len(expected)-t
        denom=1.25*t+f+.25*miss
        scores.append(1 if not expected and not accepted else 1.25*t/denom if denom else 0)
        tp+=t;fp+=f;found+=len(expected&candidates);total+=len(expected)
        singletons+=int(not expected);singleton_correct+=int(not expected and not accepted)
        for rid in lists[i]:
            eid,name,address,country=records[rid]
            if eid in expected and not str(address).strip():
                blank_found+=1;blank_accepted+=int(eid in accepted)
            if eid in accepted and eid not in expected:
                fp_rows.append(dict(policy=policy,source1_id=sid,source1_name=frame.iloc[i].business_name,
                    source1_address=frame.iloc[i].business_address,candidate_id=eid,candidate_name=name,
                    candidate_address=address,model_score=probability[i,rid],name_cosine=name_lookup.get((i,rid))))
        if policy=='baseline20_plus_name20':
            base_set=set(policies['baseline20'][i])
            for rid in lists[i]:
                if rid in base_set:continue
                right=records[rid]
                link_audit.append(dict(source1_id=sid,source1_name=frame.iloc[i].business_name,
                    source1_address=frame.iloc[i].business_address,candidate_id=right[0],candidate_name=right[1],
                    candidate_address=right[2],is_true=right[0] in expected,
                    name_cosine=name_lookup.get((i,rid)),combined_cosine=combined[i,rid],
                    model_score=probability[i,rid],accepted=probability[i,rid]>=threshold))
    blank_ids=set(missing_pool.entity_id)
    blank_total=sum(len(truth[sid]&blank_ids) for sid in frame.entity_id)
    summaries.append(dict(policy=policy,true_links=total,retrieved_true=found,recall=found/max(total,1),
        macro_F05=float(np.mean(scores)),true_accepted=tp,false_accepted=fp,
        missed_search=total-found,retrieved_rejected=found-tp,
        mean_candidates=float(np.mean([len(x) for x in lists])),max_candidates=max(map(len,lists)),
        singletons=singletons,singletons_correct=singleton_correct))
    slices.append(dict(policy=policy,blank_address_true_links=blank_total,retrieved=blank_found,
                       accepted=blank_accepted,recall=blank_found/max(blank_total,1)))
summary=pd.DataFrame(summaries)
summary['extra_true_retrieved']=summary.retrieved_true-summary.iloc[0].retrieved_true
summary.to_csv(OUT/'policy_comparison.csv',index=False)
pd.DataFrame(slices).to_csv(OUT/'missing_address_results.csv',index=False)
pd.DataFrame(link_audit).to_csv(OUT/'additional_candidates.tsv',sep='\t',index=False)
pd.DataFrame(fp_rows).to_csv(OUT/'false_accepts.tsv',sep='\t',index=False)
print('RETRIEVAL AND MATCHER RESULTS')
display(summary)
print('MISSING-ADDRESS TRUE MATCHES')
display(pd.DataFrame(slices))
print('Candidate unions may exceed 20; their counts are explicitly measured. The fixed-budget policy stays at or below 20.')
print('If recall rises but accepted true matches do not, the existing classifier is the remaining blocker. No cutoff was tuned in this experiment.')
searcher.close()
shutil.copy2(cp,OUT/'configuration.json')
archive=ROOT/'name_only_rescue_results.zip'
with zipfile.ZipFile(archive,'w',zipfile.ZIP_DEFLATED) as z:
    for path in sorted(OUT.iterdir()):z.write(path,path.name)
print('Download:',archive)
